# RAG System with ChatGPT API

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** system using:
- **OpenAI Embeddings** for document vectorization
- **ChromaDB** for vector storage and retrieval
- **ChatGPT API** for intelligent response generation

## What is RAG?
RAG combines the power of retrieval-based systems with generative AI. Instead of relying solely on the LLM's training data, RAG retrieves relevant information from your custom knowledge base and uses it to generate accurate, contextual responses.

## 1. Install Dependencies

In [ ]:
!pip install openai chromadb tiktoken langchain langchain-openai python-dotenv PyPDF2 --quiet

## 2. Import Libraries and Setup

In [ ]:
import os
import json
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
import tiktoken

# Load environment variables
load_dotenv()

# Set your OpenAI API key
# Option 1: Set in .env file as OPENAI_API_KEY=your-key-here
# Option 2: Set directly (not recommended for production)
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

print("Libraries imported successfully!")

## 3. RAG System Class

This is the core of our RAG implementation. It handles:
- Document chunking with overlap
- Embedding generation using OpenAI
- Vector storage with ChromaDB
- Semantic search and retrieval
- Response generation with ChatGPT

In [ ]:
class ChatGPTRAG:
    """
    A complete RAG (Retrieval-Augmented Generation) system using ChatGPT API.
    """
    
    def __init__(
        self,
        collection_name: str = "rag_documents",
        embedding_model: str = "text-embedding-3-small",
        chat_model: str = "gpt-3.5-turbo",
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
        persist_directory: str = "./chroma_db"
    ):
        """
        Initialize the RAG system.
        
        Args:
            collection_name: Name for the ChromaDB collection
            embedding_model: OpenAI embedding model to use
            chat_model: ChatGPT model for response generation
            chunk_size: Maximum characters per chunk
            chunk_overlap: Overlap between consecutive chunks
            persist_directory: Directory to persist ChromaDB
        """
        self.embedding_model = embedding_model
        self.chat_model = chat_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        # Initialize OpenAI client
        self.client = OpenAI()
        
        # Initialize ChromaDB with OpenAI embeddings
        self.chroma_client = chromadb.PersistentClient(path=persist_directory)
        
        # Create embedding function using OpenAI
        self.embedding_function = embedding_functions.OpenAIEmbeddingFunction(
            api_key=os.environ.get("OPENAI_API_KEY"),
            model_name=embedding_model
        )
        
        # Get or create collection
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function,
            metadata={"hnsw:space": "cosine"}
        )
        
        # Token counter for context management
        self.encoding = tiktoken.encoding_for_model(chat_model)
        
        print(f"RAG System initialized!")
        print(f"  - Embedding model: {embedding_model}")
        print(f"  - Chat model: {chat_model}")
        print(f"  - Collection: {collection_name}")
        print(f"  - Documents in collection: {self.collection.count()}")
    
    def chunk_text(self, text: str, metadata: Dict = None) -> List[Dict]:
        """
        Split text into overlapping chunks for better retrieval.
        
        Args:
            text: The text to chunk
            metadata: Optional metadata to attach to chunks
            
        Returns:
            List of chunk dictionaries with text and metadata
        """
        if metadata is None:
            metadata = {}
        
        chunks = []
        start = 0
        chunk_id = 0
        
        while start < len(text):
            # Find the end of the chunk
            end = start + self.chunk_size
            
            # If not at the end, try to break at a sentence boundary
            if end < len(text):
                # Look for sentence endings
                for sep in ['. ', '! ', '? ', '\n\n', '\n']:
                    last_sep = text.rfind(sep, start, end)
                    if last_sep != -1 and last_sep > start + self.chunk_size // 2:
                        end = last_sep + len(sep)
                        break
            
            chunk_text = text[start:end].strip()
            
            if chunk_text:
                chunk_metadata = metadata.copy()
                chunk_metadata['chunk_id'] = chunk_id
                chunk_metadata['char_start'] = start
                chunk_metadata['char_end'] = end
                
                chunks.append({
                    'text': chunk_text,
                    'metadata': chunk_metadata
                })
                chunk_id += 1
            
            # Move start position with overlap
            start = end - self.chunk_overlap if end < len(text) else len(text)
        
        return chunks
    
    def add_documents(self, documents: List[Dict[str, Any]]) -> int:
        """
        Add documents to the vector store.
        
        Args:
            documents: List of dicts with 'text' and optional 'metadata' keys
            
        Returns:
            Number of chunks added
        """
        all_chunks = []
        
        for doc in documents:
            text = doc.get('text', '')
            metadata = doc.get('metadata', {})
            chunks = self.chunk_text(text, metadata)
            all_chunks.extend(chunks)
        
        if not all_chunks:
            print("No chunks to add.")
            return 0
        
        # Prepare data for ChromaDB
        ids = [f"chunk_{self.collection.count() + i}" for i in range(len(all_chunks))]
        texts = [chunk['text'] for chunk in all_chunks]
        metadatas = [chunk['metadata'] for chunk in all_chunks]
        
        # Add to collection
        self.collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas
        )
        
        print(f"Added {len(all_chunks)} chunks to the collection.")
        print(f"Total documents in collection: {self.collection.count()}")
        
        return len(all_chunks)
    
    def add_text(self, text: str, metadata: Dict = None) -> int:
        """
        Convenience method to add a single text document.
        
        Args:
            text: Text content to add
            metadata: Optional metadata
            
        Returns:
            Number of chunks added
        """
        return self.add_documents([{'text': text, 'metadata': metadata or {}}])
    
    def retrieve(self, query: str, n_results: int = 5) -> List[Dict]:
        """
        Retrieve relevant documents for a query.
        
        Args:
            query: The search query
            n_results: Number of results to retrieve
            
        Returns:
            List of retrieved documents with scores
        """
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results,
            include=['documents', 'metadatas', 'distances']
        )
        
        retrieved_docs = []
        for i in range(len(results['documents'][0])):
            retrieved_docs.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i]
            })
        
        return retrieved_docs
    
    def count_tokens(self, text: str) -> int:
        """Count tokens in text."""
        return len(self.encoding.encode(text))
    
    def generate_response(
        self,
        query: str,
        n_results: int = 5,
        max_context_tokens: int = 3000,
        system_prompt: str = None,
        temperature: float = 0.7,
        include_sources: bool = True
    ) -> Dict[str, Any]:
        """
        Generate a response using RAG (retrieve + generate).
        
        Args:
            query: User's question
            n_results: Number of documents to retrieve
            max_context_tokens: Maximum tokens for context
            system_prompt: Custom system prompt
            temperature: Response creativity (0-2)
            include_sources: Whether to return source documents
            
        Returns:
            Dictionary with response and metadata
        """
        # Retrieve relevant documents
        retrieved_docs = self.retrieve(query, n_results)
        
        if not retrieved_docs:
            return {
                'response': "I don't have any relevant information to answer your question.",
                'sources': [],
                'context_used': 0
            }
        
        # Build context from retrieved documents
        context_parts = []
        total_tokens = 0
        used_docs = []
        
        for doc in retrieved_docs:
            doc_text = doc['text']
            doc_tokens = self.count_tokens(doc_text)
            
            if total_tokens + doc_tokens <= max_context_tokens:
                context_parts.append(doc_text)
                total_tokens += doc_tokens
                used_docs.append(doc)
            else:
                break
        
        context = "\n\n---\n\n".join(context_parts)
        
        # Default system prompt
        if system_prompt is None:
            system_prompt = """You are a helpful assistant that answers questions based on the provided context.
            
Rules:
1. Only use information from the provided context to answer questions
2. If the context doesn't contain relevant information, say so clearly
3. Be concise but thorough in your responses
4. If you're unsure about something, express that uncertainty
5. Do not make up information that's not in the context"""
        
        # Construct the user message with context
        user_message = f"""Context Information:
---
{context}
---

Based on the above context, please answer the following question:
{query}"""
        
        # Generate response using ChatGPT
        completion = self.client.chat.completions.create(
            model=self.chat_model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=temperature,
            max_tokens=1000
        )
        
        response_text = completion.choices[0].message.content
        
        result = {
            'response': response_text,
            'tokens_used': {
                'prompt': completion.usage.prompt_tokens,
                'completion': completion.usage.completion_tokens,
                'total': completion.usage.total_tokens
            },
            'context_chunks': len(used_docs)
        }
        
        if include_sources:
            result['sources'] = used_docs
        
        return result
    
    def chat(
        self,
        messages: List[Dict[str, str]],
        n_results: int = 5,
        system_prompt: str = None
    ) -> Dict[str, Any]:
        """
        Multi-turn chat with RAG support.
        
        Args:
            messages: List of message dicts with 'role' and 'content'
            n_results: Number of documents to retrieve
            system_prompt: Custom system prompt
            
        Returns:
            Response dictionary
        """
        if not messages:
            return {'response': 'No messages provided.'}
        
        # Get the last user message for retrieval
        last_user_message = None
        for msg in reversed(messages):
            if msg.get('role') == 'user':
                last_user_message = msg.get('content', '')
                break
        
        if not last_user_message:
            return {'response': 'No user message found.'}
        
        # Retrieve context based on the last user message
        retrieved_docs = self.retrieve(last_user_message, n_results)
        
        # Build context
        context_parts = [doc['text'] for doc in retrieved_docs[:3]]  # Use top 3
        context = "\n\n---\n\n".join(context_parts)
        
        # Default system prompt with context
        if system_prompt is None:
            system_prompt = f"""You are a helpful assistant that answers questions based on the provided knowledge base.

Knowledge Base Context:
---
{context}
---

Rules:
1. Use the knowledge base context to answer questions when relevant
2. Be conversational and helpful
3. If the context doesn't have relevant information, acknowledge that
4. Maintain conversation continuity"""
        
        # Prepare messages for API
        api_messages = [{"role": "system", "content": system_prompt}]
        api_messages.extend(messages)
        
        # Generate response
        completion = self.client.chat.completions.create(
            model=self.chat_model,
            messages=api_messages,
            temperature=0.7,
            max_tokens=1000
        )
        
        return {
            'response': completion.choices[0].message.content,
            'sources': retrieved_docs,
            'tokens_used': completion.usage.total_tokens
        }
    
    def clear_collection(self):
        """Clear all documents from the collection."""
        # Get all IDs and delete them
        all_ids = self.collection.get()['ids']
        if all_ids:
            self.collection.delete(ids=all_ids)
        print(f"Collection cleared. Current count: {self.collection.count()}")
    
    def get_collection_stats(self) -> Dict:
        """Get statistics about the collection."""
        return {
            'total_documents': self.collection.count(),
            'embedding_model': self.embedding_model,
            'chat_model': self.chat_model,
            'chunk_size': self.chunk_size,
            'chunk_overlap': self.chunk_overlap
        }

## 4. Helper Functions

Utility functions for loading different types of documents.

In [ ]:
def load_text_file(file_path: str) -> str:
    """Load text from a file."""
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

def load_pdf(file_path: str) -> str:
    """Load text from a PDF file."""
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text
    except ImportError:
        print("PyPDF2 not installed. Install with: pip install PyPDF2")
        return ""

def load_multiple_files(file_paths: List[str]) -> List[Dict]:
    """Load multiple files into document format."""
    documents = []
    for path in file_paths:
        if path.endswith('.pdf'):
            text = load_pdf(path)
        else:
            text = load_text_file(path)
        
        documents.append({
            'text': text,
            'metadata': {'source': path}
        })
    return documents

print("Helper functions loaded!")

## 5. Initialize the RAG System

Create an instance of the RAG system. Make sure your OpenAI API key is set!

In [ ]:
# Initialize the RAG system
rag = ChatGPTRAG(
    collection_name="my_knowledge_base",
    embedding_model="text-embedding-3-small",  # Cost-effective embedding model
    chat_model="gpt-3.5-turbo",  # Or "gpt-4" for better quality
    chunk_size=1000,
    chunk_overlap=200
)

## 6. Add Documents to Knowledge Base

Let's add some sample documents to demonstrate the system.

In [ ]:
# Sample documents about artificial intelligence
sample_documents = [
    {
        'text': """
        Machine Learning Fundamentals
        
        Machine learning is a subset of artificial intelligence that enables systems to learn and improve 
        from experience without being explicitly programmed. The primary aim is to allow computers to 
        learn automatically without human intervention or assistance and adjust actions accordingly.
        
        Types of Machine Learning:
        
        1. Supervised Learning: The algorithm learns from labeled training data, and makes predictions 
        based on that data. Common applications include spam detection, image classification, and 
        price prediction. Examples include Linear Regression, Decision Trees, and Neural Networks.
        
        2. Unsupervised Learning: The algorithm learns patterns from unlabeled data. It's used for 
        clustering, dimensionality reduction, and association. Common algorithms include K-means 
        clustering, Principal Component Analysis (PCA), and Autoencoders.
        
        3. Reinforcement Learning: The algorithm learns by interacting with an environment, receiving 
        rewards or penalties for actions. It's used in game playing, robotics, and autonomous vehicles. 
        Notable examples include Q-Learning and Deep Q-Networks (DQN).
        
        4. Semi-supervised Learning: Combines a small amount of labeled data with a large amount of 
        unlabeled data during training. This approach is useful when labeling data is expensive.
        """,
        'metadata': {'topic': 'machine_learning', 'subtopic': 'fundamentals'}
    },
    {
        'text': """
        Natural Language Processing (NLP)
        
        Natural Language Processing is a branch of AI that helps computers understand, interpret, 
        and manipulate human language. NLP draws from many disciplines, including computer science 
        and computational linguistics.
        
        Key NLP Tasks:
        
        - Text Classification: Assigning categories to text documents (sentiment analysis, spam detection)
        - Named Entity Recognition (NER): Identifying and classifying named entities in text
        - Part-of-Speech Tagging: Assigning grammatical tags to words
        - Machine Translation: Translating text from one language to another
        - Question Answering: Automatically answering questions based on text
        - Text Summarization: Creating concise summaries of longer documents
        - Sentiment Analysis: Determining the emotional tone of text
        
        Modern NLP Techniques:
        
        Transformer architecture has revolutionized NLP. Models like BERT, GPT, and T5 have achieved 
        state-of-the-art results on many NLP benchmarks. These models use self-attention mechanisms 
        to process text more effectively than previous RNN-based approaches.
        
        Large Language Models (LLMs) like GPT-4, Claude, and Llama have demonstrated remarkable 
        capabilities in understanding context, generating human-like text, and performing various 
        language tasks with minimal task-specific training.
        """,
        'metadata': {'topic': 'nlp', 'subtopic': 'overview'}
    },
    {
        'text': """
        Retrieval-Augmented Generation (RAG)
        
        RAG is an AI framework that combines the capabilities of retrieval systems with generative 
        language models. It addresses key limitations of traditional LLMs by providing access to 
        external knowledge bases.
        
        How RAG Works:
        
        1. Document Ingestion: Documents are processed and split into chunks
        2. Embedding Generation: Each chunk is converted into a vector representation
        3. Vector Storage: Embeddings are stored in a vector database for efficient retrieval
        4. Query Processing: User queries are embedded using the same model
        5. Retrieval: Most similar documents are retrieved based on vector similarity
        6. Generation: Retrieved context is provided to the LLM to generate accurate responses
        
        Benefits of RAG:
        
        - Reduces hallucinations by grounding responses in retrieved facts
        - Enables access to up-to-date or proprietary information
        - Provides transparency through source citations
        - More cost-effective than fine-tuning for domain adaptation
        - Allows easy updates to the knowledge base without retraining
        
        Common Components:
        
        - Vector Databases: ChromaDB, Pinecone, Weaviate, FAISS, Milvus
        - Embedding Models: OpenAI embeddings, Sentence Transformers, Cohere
        - LLMs: GPT-4, Claude, Llama, Mistral
        - Orchestration: LangChain, LlamaIndex
        """,
        'metadata': {'topic': 'rag', 'subtopic': 'architecture'}
    },
    {
        'text': """
        Deep Learning and Neural Networks
        
        Deep learning is a subset of machine learning based on artificial neural networks with 
        multiple layers (hence "deep"). These networks learn hierarchical representations of data.
        
        Types of Neural Networks:
        
        1. Feedforward Neural Networks (FNN): The simplest form where information flows in one 
        direction from input to output. Used for basic classification and regression.
        
        2. Convolutional Neural Networks (CNN): Specialized for processing grid-like data such 
        as images. Uses convolutional layers to automatically learn spatial hierarchies. 
        Applications include image classification, object detection, and medical image analysis.
        
        3. Recurrent Neural Networks (RNN): Designed for sequential data with connections that 
        form directed cycles. Variants include LSTM and GRU which handle long-term dependencies 
        better. Used for time series, speech recognition, and language modeling.
        
        4. Transformers: Architecture based entirely on attention mechanisms. Processes entire 
        sequences in parallel rather than sequentially. Foundation for models like BERT and GPT. 
        Excels at NLP tasks and increasingly used in computer vision (Vision Transformers).
        
        5. Generative Adversarial Networks (GANs): Two networks (generator and discriminator) 
        compete against each other. Used for image generation, style transfer, and data augmentation.
        
        Training Deep Networks:
        
        - Backpropagation: Algorithm for computing gradients
        - Optimizers: SGD, Adam, RMSprop for updating weights
        - Regularization: Dropout, batch normalization to prevent overfitting
        - Transfer Learning: Using pre-trained models as starting point
        """,
        'metadata': {'topic': 'deep_learning', 'subtopic': 'neural_networks'}
    }
]

# Add documents to the knowledge base
rag.add_documents(sample_documents)

## 7. Query the RAG System

Now let's test our RAG system with some questions!

In [ ]:
# Test Query 1: About RAG
query1 = "What are the main benefits of using RAG?"
response1 = rag.generate_response(query1, n_results=3)

print("Question:", query1)
print("\nAnswer:")
print(response1['response'])
print(f"\nTokens used: {response1['tokens_used']}")
print(f"Context chunks used: {response1['context_chunks']}")

In [ ]:
# Test Query 2: About Machine Learning
query2 = "Explain the different types of machine learning"
response2 = rag.generate_response(query2, n_results=3)

print("Question:", query2)
print("\nAnswer:")
print(response2['response'])

In [ ]:
# Test Query 3: About Deep Learning
query3 = "What is the difference between CNN and RNN?"
response3 = rag.generate_response(query3, n_results=3)

print("Question:", query3)
print("\nAnswer:")
print(response3['response'])

## 8. View Retrieved Sources

Let's see what documents were retrieved for a query.

In [ ]:
# Check retrieved sources for transparency
query = "How do transformers work in NLP?"
retrieved = rag.retrieve(query, n_results=3)

print(f"Query: {query}\n")
print("Retrieved Documents:")
print("=" * 50)

for i, doc in enumerate(retrieved, 1):
    print(f"\n--- Document {i} ---")
    print(f"Distance (lower is better): {doc['distance']:.4f}")
    print(f"Metadata: {doc['metadata']}")
    print(f"\nContent Preview (first 300 chars):")
    print(doc['text'][:300] + "...")

## 9. Multi-turn Chat with RAG

You can also have a conversation with the RAG system.

In [ ]:
# Multi-turn conversation example
conversation = []

# First turn
user_msg1 = "What is supervised learning?"
conversation.append({"role": "user", "content": user_msg1})
response = rag.chat(conversation)
print(f"User: {user_msg1}")
print(f"Assistant: {response['response']}\n")
conversation.append({"role": "assistant", "content": response['response']})

# Second turn (follow-up question)
user_msg2 = "Can you give me some specific examples of where it's used?"
conversation.append({"role": "user", "content": user_msg2})
response = rag.chat(conversation)
print(f"User: {user_msg2}")
print(f"Assistant: {response['response']}")

## 10. Add Your Own Documents

You can add your own documents to create a custom knowledge base.

In [ ]:
# Example: Add a custom document
custom_text = """
Your custom document content goes here. This could be:
- Company documentation
- Research papers
- Product manuals
- Any text-based knowledge you want the system to learn

The RAG system will chunk this text, create embeddings, and store them
for retrieval when answering questions.
"""

# Add with metadata
rag.add_text(
    custom_text, 
    metadata={'source': 'custom', 'type': 'example', 'date': '2024-01-01'}
)

# Check collection stats
print("\nCollection Statistics:")
print(rag.get_collection_stats())

In [ ]:
# Example: Load from files (uncomment to use)
# files = ['document1.txt', 'document2.pdf']
# docs = load_multiple_files(files)
# rag.add_documents(docs)

## 11. Advanced: Custom System Prompts

Customize how the assistant responds by modifying the system prompt.

In [ ]:
# Custom system prompt for specific use case
technical_prompt = """You are a technical expert assistant specializing in AI and machine learning.

When answering questions:
1. Use the provided context as your primary source
2. Explain concepts clearly with technical accuracy
3. Include relevant examples when appropriate
4. If something is not in the context, clearly state that
5. Use bullet points and structured formatting for clarity"""

query = "Compare supervised and unsupervised learning"
response = rag.generate_response(
    query,
    system_prompt=technical_prompt,
    temperature=0.3  # Lower temperature for more focused responses
)

print("Question:", query)
print("\nAnswer:")
print(response['response'])

## 12. Interactive Chat Interface (Optional)

Run this cell for an interactive chat experience.

In [ ]:
def interactive_chat():
    """Simple interactive chat interface."""
    print("RAG Chat Interface")
    print("Type 'quit' to exit, 'clear' to reset conversation")
    print("=" * 50)
    
    conversation = []
    
    while True:
        user_input = input("\nYou: ").strip()
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        elif user_input.lower() == 'clear':
            conversation = []
            print("Conversation cleared.")
            continue
        elif not user_input:
            continue
        
        conversation.append({"role": "user", "content": user_input})
        
        try:
            response = rag.chat(conversation)
            assistant_msg = response['response']
            conversation.append({"role": "assistant", "content": assistant_msg})
            print(f"\nAssistant: {assistant_msg}")
        except Exception as e:
            print(f"\nError: {e}")

# Uncomment to start interactive chat
# interactive_chat()

## 13. Best Practices and Tips

### Optimizing RAG Performance:

1. **Chunk Size**: 
   - Smaller chunks (500-1000 chars) for specific retrieval
   - Larger chunks (1500-2000 chars) for more context

2. **Chunk Overlap**: 
   - 10-20% overlap helps maintain context between chunks

3. **Number of Retrieved Documents**:
   - Start with 3-5 documents
   - Increase if answers lack detail
   - Decrease if responses are unfocused

4. **Embedding Models**:
   - `text-embedding-3-small`: Cost-effective, good performance
   - `text-embedding-3-large`: Better quality, higher cost

5. **Chat Models**:
   - `gpt-3.5-turbo`: Fast, cost-effective
   - `gpt-4`: Better reasoning, higher cost
   - `gpt-4-turbo`: Balance of quality and speed

### Common Issues:

- **Irrelevant retrieval**: Improve chunk boundaries or add more documents
- **Hallucinations**: Lower temperature, strengthen system prompt
- **Incomplete answers**: Increase n_results or chunk size
- **High costs**: Use smaller models, optimize chunk sizes

## 14. Cleanup (Optional)

Clear the database if you want to start fresh.

In [ ]:
# Uncomment to clear the collection
# rag.clear_collection()

## Next Steps

1. **Add your own documents**: Load PDFs, text files, or any documents relevant to your use case
2. **Customize prompts**: Tailor the system prompt for your specific domain
3. **Experiment with parameters**: Try different chunk sizes, overlap, and retrieval counts
4. **Integrate evaluation**: Add metrics to measure response quality
5. **Scale up**: Use cloud-hosted vector databases for production

Happy building! 🚀